# ScreamingFace Phase 1 · discover, load, author

Connect the SDK to the temporary `screamingface-engine` development profile, inspect what the engine
advertises, and construct the immutable values used by later execution phases.

**This is the Phase 1 discovery walkthrough—not the product quickstart.** It covers engine
configuration, model and benchmark discovery, benchmark loading, local benchmark definitions,
and Fusion authoring. It deliberately does not call the model routes. SDK Fusion execution now
exists in Phase 2C, but remains outside this discovery-focused walkthrough.

## Before you run it

From the repository root, start the tracked local stack:

```bash
cd packages/screamingface/apps/screamingface-engine
./dev.sh
```

This starts the URL4 engine at `http://127.0.0.1:4404` and AI Gateway at
`http://127.0.0.1:9105`. Discovery and benchmark loading use only the URL4 engine; only an
executable model route contacts AI Gateway. Only one development stack can own those ports, so
stop any earlier URL4 or AI Gateway containers before running `./dev.sh`.

The published GPQA and DRACO case routes read their canonical Hugging Face datasets. Before
loading one, accept any dataset terms and expose your token to Compose:

```bash
huggingface-cli login
export HF_TOKEN=hf_...
```

There is no synthetic or in-process fallback.

## 1 · Point the SDK at the engine

In [ ]:
import json
import os

import httpx

import screamingface as sf

ENGINE_URL = os.environ.get("SCREAMINGFACE_ENGINE_URL", "http://127.0.0.1:4404")
sf.config(engine=ENGINE_URL)

# This is currently also the SDK default, so configuration is optional locally.
httpx.get(f"{ENGINE_URL}/healthz", timeout=5).text

`sf.config(...)` only stores and validates the URL. It does not perform a request.
The health check above is the first network call in this notebook. A future hosted deployment can
be selected with the same API:

```python
sf.config(engine="https://url4.example")
```

## 2 · Inspect the ScreamingFace registry

In [ ]:
registry_response = httpx.get(f"{ENGINE_URL}/.well-known/screamingface", timeout=5)
registry_response.raise_for_status()

{
    "content_type": registry_response.headers["content-type"],
    "plaintext_preview": registry_response.text[:120] + "…",
}

In [ ]:
registry = json.loads(registry_response.text)
registry

The URL4 node returns plaintext. The SDK parses that text and validates the complete
`screamingface.registry.v1` shape before exposing IDs. Model entries describe executable route
identities and supported tools. Benchmark entries point to versioned manifests and case streams.

## 3 · Discover available models and benchmarks

In [ ]:
{
    "all_models": sf.models.list(),
    "gemini_models": sf.models.list(query="gemini"),
    "web_search_models": sf.models.list(tools=["web_search"]),
    "benchmarks": sf.benchmarks.list(),
    "research_benchmarks": sf.benchmarks.list(tools=["web_search"]),
}

Discovery returns canonical IDs in registry order. It does not return a static
catalog bundled in the SDK, and it does not infer provider access or authentication. The
configured engine is the source of truth.

Phase 2A intentionally advertises no `web_search` models until a real named-tool adapter exists,
so `web_search_models` is empty. DRACO remains discoverable through `research_benchmarks` because
its manifest truthfully declares that still-unmet execution requirement.

## 4 · Load a published benchmark when dataset access is ready

In [ ]:
# Change this after the engine container has access to your Hugging Face token.
LOAD_REMOTE_BENCHMARK = False

benchmark = sf.benchmarks.load("gpqa@1") if LOAD_REMOTE_BENCHMARK else None

benchmark or (
    "Set LOAD_REMOTE_BENCHMARK = True to fetch, parse, and validate the GPQA "
    "manifest and case stream."
)

`sf.benchmarks.load("gpqa@1")` is eager:

1. fetch and validate the engine registry;
2. resolve the benchmark's same-engine manifest route;
3. validate its grader, aggregator, tools, and case-stream contract;
4. fetch and parse every normalized NDJSON case; and
5. return one immutable `sf.Benchmark`.

Malformed manifests, duplicate case IDs, unknown judge models, and HTTP failures remain typed
errors. Loading never calls a panel model or AI Gateway.

## 5 · Define a small benchmark in ordinary Python

In [ ]:
arithmetic = sf.Benchmark(
    "arithmetic-smoke-test",
    title="Arithmetic smoke test",
    cases=[
        sf.Case(
            "addition",
            "What is 2 + 2?\n\nA. 3\nB. 4\n\nReply with only A or B.",
            reference="B",
            metadata={"subject": "arithmetic"},
        ),
        sf.Case(
            "multiplication",
            "What is 3 × 3?\n\nA. 9\nB. 6\n\nReply with only A or B.",
            reference="A",
            metadata={"subject": "arithmetic"},
        ),
    ],
    grader=sf.graders.ExactChoice(),
    aggregator=sf.aggregators.Mean(),
)
arithmetic

Dataset cleaning stays in ordinary Python. ScreamingFace begins at stable `sf.Case`
values: one input, a sealed JSON reference used only for grading, and optional reporting metadata.
The SDK intentionally does not add an ETL language or case-browser abstraction.

## 6 · Author a Fusion without executing it

In [ ]:
fusion = sf.Fusion(
    "frontier-trio",
    models=[
        "codex/gpt-5.5",
        "gemini/2.5",
        "claude/sonnet-4.6",
    ],
    prompt="Answer the question carefully: $question",
    reducer=sf.reducers.MajorityVote(),
)

{
    "name": fusion.name,
    "models": fusion.models,
    "model_ids": fusion.model_ids,
    "reducer": fusion.reducer,
}

Construction is local and network-free. Strings are the concise model form; a model
mapping can add a per-member `prompt` or scalar `params`. Reducers are typed strategies under
`sf.reducers`; graders and aggregators follow the same namespaced convention.

The public values in this notebook are immutable. That makes benchmark definitions and Fusion
recipes safe to inspect and pass between later execution stages.

## Phase 1 boundary

You have now exercised everything Phase 1 promises:

- configure one URL4 engine;
- inspect and validate its ScreamingFace registry;
- list models and benchmarks with filters;
- optionally load a published benchmark and its real cases;
- define a local benchmark with the same public types; and
- author an immutable Fusion.

This walkthrough deliberately stops before `fusion.run(...)`. Phase 2B supplies persistent
tool-free model routes, the deterministic majority-vote route, and `GET /v1?q=...`; Phase 2C now
supplies SDK URL4 compilation, plaintext result validation, and in-memory run results. Grading,
aggregation, and `fusion.evaluate(...)` remain Phase 3.
The SDK will continue to contact only the URL4 engine; only the engine's model adapter may contact
AI Gateway.